# Generation and pairwise file construction

Executed locally on an RTX 4050 using the repository-local environment and existing training artifacts. The saved outputs below are the actual execution results.


In [1]:
import os
from pathlib import Path
import shlex
import subprocess
import sys


def find_repo_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "scripts").is_dir() and (candidate / "alignment").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the cs5326-pa2 repository.")


REPO_ROOT = find_repo_root()
SCRIPTS = REPO_ROOT / "scripts"
OUTPUTS = REPO_ROOT / "outputs"
PYTHON = Path(sys.executable).resolve()

environment_name = Path(sys.prefix).name.lower()
if environment_name not in {"torch_env", ".venv-pa2"}:
    raise RuntimeError(
        f"Wrong Python environment: {sys.prefix}. "
        "Select the 'Python (cs5326-pa2 local)' Jupyter kernel."
    )

PROCESS_ENV = os.environ.copy()
PROCESS_ENV["CUDA_VISIBLE_DEVICES"] = "0"
PROCESS_ENV["PYTHONPATH"] = str(REPO_ROOT)


def run_script(script_name, *arguments):
    command = [str(PYTHON), str(SCRIPTS / script_name), *map(str, arguments)]
    print("Running:", shlex.join(command), flush=True)
    subprocess.run(command, cwd=REPO_ROOT, env=PROCESS_ENV, check=True)


os.chdir(REPO_ROOT)
print("Repository:", REPO_ROOT)
print("Python:", PYTHON)
print("CUDA visibility for child processes:", PROCESS_ENV["CUDA_VISIBLE_DEVICES"])

Repository: D:\projects\Assignments\CS5326\PA2\cs5326-pa2
Python: D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe
CUDA visibility for child processes: 0


In [2]:
run_script("check_environment.py", "--output_path", "outputs/environment_local.json")

Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\check_environment.py' --output_path outputs/environment_local.json


In [6]:
TRAIN_OUTPUTS = OUTPUTS
DECODING = [
    "--max_new_tokens", "256",
    "--batch_size", "256",
    "--repetition_penalty", "1.0",
    "--no_repeat_ngram_size", "0",
    "--device", "cuda:0",
    "--dtype", "float16",
]

required_models = [
    TRAIN_OUTPUTS / "sft" / "best",
    TRAIN_OUTPUTS / "dpo_uf" / "best",
    TRAIN_OUTPUTS / "dpo_hh" / "best",
    TRAIN_OUTPUTS / "lora_dpo" / "merged",
]
missing = [str(path) for path in required_models if not path.is_dir()]
if missing:
    raise FileNotFoundError(f"Missing training artifacts: {missing}")
print("Training artifacts: OK")

Training artifacts: OK


## Task 2: Fixed AlpacaEval subset

In [4]:
run_script(
    "make_alpaca_subset.py",
    "--output_path", "eval_sets/alpaca_eval_300_seed42.json",
    "--size", "300",
    "--seed", "42",
)

Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\make_alpaca_subset.py' --output_path eval_sets/alpaca_eval_300_seed42.json --size 300 --seed 42


## Task 6: Sanity check on three prompts

In [5]:
run_script(
    "generate.py",
    "--model_path", TRAIN_OUTPUTS / "sft" / "best",
    "--prompts_path", "eval_sets/alpaca_eval_300_seed42.json",
    "--output_path", "outputs/sanity/sft_alpaca.json",
    "--generator_name", "sft",
    "--limit", "3",
    "--print_outputs",
    *DECODING,
)

Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\generate.py' --model_path 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\outputs\sft\best' --prompts_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/sanity/sft_alpaca.json --generator_name sft --limit 3 --print_outputs --max_new_tokens 256 --batch_size 4 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --device cuda:0 --dtype float16


In [6]:
run_script(
    "generate.py",
    "--model_path", TRAIN_OUTPUTS / "dpo_uf" / "best",
    "--prompts_path", "eval_sets/alpaca_eval_300_seed42.json",
    "--output_path", "outputs/sanity/dpo_uf_alpaca.json",
    "--generator_name", "dpo_uf",
    "--limit", "3",
    "--print_outputs",
    *DECODING,
)

Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\generate.py' --model_path 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\outputs\dpo_uf\best' --prompts_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/sanity/dpo_uf_alpaca.json --generator_name dpo_uf --limit 3 --print_outputs --max_new_tokens 256 --batch_size 4 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --device cuda:0 --dtype float16


In [7]:
run_script(
    "generate.py",
    "--model_path", TRAIN_OUTPUTS / "dpo_hh" / "best",
    "--prompts_path", "eval_sets/hh_rlhf_eval_300_seed42.json",
    "--output_path", "outputs/sanity/dpo_hh.json",
    "--generator_name", "dpo_hh",
    "--limit", "3",
    "--print_outputs",
    *DECODING,
)

Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\generate.py' --model_path 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\outputs\dpo_hh\best' --prompts_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/sanity/dpo_hh.json --generator_name dpo_hh --limit 3 --print_outputs --max_new_tokens 256 --batch_size 4 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --device cuda:0 --dtype float16


In [8]:
run_script(
    "generate.py",
    "--model_path", TRAIN_OUTPUTS / "lora_dpo" / "merged",
    "--prompts_path", "eval_sets/persona_style_eval_40.json",
    "--output_path", "outputs/sanity/lora_persona.json",
    "--generator_name", "lora_dpo",
    "--dataset_name", "persona_style_eval",
    "--prompt_format", "chat",
    "--limit", "3",
    "--print_outputs",
    *DECODING,
)

Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\generate.py' --model_path 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\outputs\lora_dpo\merged' --prompts_path eval_sets/persona_style_eval_40.json --output_path outputs/sanity/lora_persona.json --generator_name lora_dpo --dataset_name persona_style_eval --prompt_format chat --limit 3 --print_outputs --max_new_tokens 256 --batch_size 4 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --device cuda:0 --dtype float16


## Task 2 and Task 6: Full generation on AlpacaEval

In [4]:
run_script(
    "generate.py",
    "--model_path", "Qwen/Qwen2.5-0.5B",
    "--prompts_path", "eval_sets/alpaca_eval_300_seed42.json",
    "--output_path", "outputs/generations/base_alpaca.json",
    "--generator_name", "qwen2.5-0.5b-base",
    "--dataset_name", "alpaca_eval",
    *DECODING,
)

Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\generate.py' --model_path Qwen/Qwen2.5-0.5B --prompts_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/base_alpaca.json --generator_name qwen2.5-0.5b-base --dataset_name alpaca_eval --max_new_tokens 256 --batch_size 64 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --device cuda:0 --dtype float16


In [7]:
run_script(
    "generate.py",
    "--model_path", TRAIN_OUTPUTS / "sft" / "best",
    "--prompts_path", "eval_sets/alpaca_eval_300_seed42.json",
    "--output_path", "outputs/generations/sft_alpaca.json",
    "--generator_name", "sft",
    "--dataset_name", "alpaca_eval",
    *DECODING,
)

Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\generate.py' --model_path 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\outputs\sft\best' --prompts_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/sft_alpaca.json --generator_name sft --dataset_name alpaca_eval --max_new_tokens 256 --batch_size 256 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --device cuda:0 --dtype float16


In [8]:
run_script(
    "generate.py",
    "--model_path", TRAIN_OUTPUTS / "dpo_uf" / "best",
    "--prompts_path", "eval_sets/alpaca_eval_300_seed42.json",
    "--output_path", "outputs/generations/dpo_uf_alpaca.json",
    "--generator_name", "dpo_uf",
    "--dataset_name", "alpaca_eval",
    *DECODING,
)

Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\generate.py' --model_path 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\outputs\dpo_uf\best' --prompts_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/dpo_uf_alpaca.json --generator_name dpo_uf --dataset_name alpaca_eval --max_new_tokens 256 --batch_size 256 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --device cuda:0 --dtype float16


## Generation on the synthetic helpful/harmless set

In [9]:
run_script(
    "generate.py",
    "--model_path", TRAIN_OUTPUTS / "sft" / "best",
    "--prompts_path", "eval_sets/hh_rlhf_eval_300_seed42.json",
    "--output_path", "outputs/generations/sft_hh.json",
    "--generator_name", "sft",
    "--dataset_name", "hh_rlhf_style_eval",
    *DECODING,
)

Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\generate.py' --model_path 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\outputs\sft\best' --prompts_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/generations/sft_hh.json --generator_name sft --dataset_name hh_rlhf_style_eval --max_new_tokens 256 --batch_size 256 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --device cuda:0 --dtype float16


In [10]:
run_script(
    "generate.py",
    "--model_path", TRAIN_OUTPUTS / "dpo_hh" / "best",
    "--prompts_path", "eval_sets/hh_rlhf_eval_300_seed42.json",
    "--output_path", "outputs/generations/dpo_hh_hh.json",
    "--generator_name", "dpo_hh",
    "--dataset_name", "hh_rlhf_style_eval",
    *DECODING,
)

Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\generate.py' --model_path 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\outputs\dpo_hh\best' --prompts_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/generations/dpo_hh_hh.json --generator_name dpo_hh --dataset_name hh_rlhf_style_eval --max_new_tokens 256 --batch_size 256 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --device cuda:0 --dtype float16


In [11]:
run_script(
    "generate.py",
    "--model_path", "Qwen/Qwen2.5-0.5B",
    "--prompts_path", "eval_sets/hh_rlhf_eval_300_seed42.json",
    "--output_path", "outputs/generations/base_hh.json",
    "--generator_name", "qwen2.5-0.5b-base",
    "--dataset_name", "hh_rlhf_style_eval",
    *DECODING,
)

Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\generate.py' --model_path Qwen/Qwen2.5-0.5B --prompts_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/generations/base_hh.json --generator_name qwen2.5-0.5b-base --dataset_name hh_rlhf_style_eval --max_new_tokens 256 --batch_size 256 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --device cuda:0 --dtype float16


## Generation for the LoRA persona model

In [12]:
run_script(
    "generate.py",
    "--model_path", "HuggingFaceTB/SmolLM2-360M-Instruct",
    "--prompts_path", "eval_sets/persona_style_eval_40.json",
    "--output_path", "outputs/generations/base_persona.json",
    "--generator_name", "smollm2-360m-instruct",
    "--dataset_name", "persona_style_eval",
    "--prompt_format", "chat",
    *DECODING,
)

Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\generate.py' --model_path HuggingFaceTB/SmolLM2-360M-Instruct --prompts_path eval_sets/persona_style_eval_40.json --output_path outputs/generations/base_persona.json --generator_name smollm2-360m-instruct --dataset_name persona_style_eval --prompt_format chat --max_new_tokens 256 --batch_size 256 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --device cuda:0 --dtype float16


In [13]:
run_script(
    "generate.py",
    "--model_path", TRAIN_OUTPUTS / "lora_dpo" / "merged",
    "--prompts_path", "eval_sets/persona_style_eval_40.json",
    "--output_path", "outputs/generations/lora_persona.json",
    "--generator_name", "lora_dpo",
    "--dataset_name", "persona_style_eval",
    "--prompt_format", "chat",
    *DECODING,
)

Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\generate.py' --model_path 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\outputs\lora_dpo\merged' --prompts_path eval_sets/persona_style_eval_40.json --output_path outputs/generations/lora_persona.json --generator_name lora_dpo --dataset_name persona_style_eval --prompt_format chat --max_new_tokens 256 --batch_size 256 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --device cuda:0 --dtype float16


## Randomized A/B pairwise files

In [14]:
run_script(
    "build_pairs.py",
    "--candidate_path", "outputs/generations/dpo_uf_alpaca.json",
    "--baseline_path", "outputs/generations/sft_alpaca.json",
    "--output_path", "outputs/eval/dpo_uf_vs_sft_pairs.jsonl",
    "--candidate_name", "dpo_uf",
    "--baseline_name", "sft",
    "--seed", "42",
)
run_script(
    "build_pairs.py",
    "--candidate_path", "outputs/generations/dpo_hh_hh.json",
    "--baseline_path", "outputs/generations/sft_hh.json",
    "--output_path", "outputs/eval/dpo_hh_vs_sft_pairs.jsonl",
    "--candidate_name", "dpo_hh",
    "--baseline_name", "sft",
    "--seed", "42",
)
run_script(
    "build_pairs.py",
    "--candidate_path", "outputs/generations/lora_persona.json",
    "--baseline_path", "outputs/generations/base_persona.json",
    "--output_path", "outputs/eval/lora_vs_base_pairs.jsonl",
    "--candidate_name", "lora_dpo",
    "--baseline_name", "smollm2-instruct",
    "--seed", "42",
)

Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\build_pairs.py' --candidate_path outputs/generations/dpo_uf_alpaca.json --baseline_path outputs/generations/sft_alpaca.json --output_path outputs/eval/dpo_uf_vs_sft_pairs.jsonl --candidate_name dpo_uf --baseline_name sft --seed 42
Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\build_pairs.py' --candidate_path outputs/generations/dpo_hh_hh.json --baseline_path outputs/generations/sft_hh.json --output_path outputs/eval/dpo_hh_vs_sft_pairs.jsonl --candidate_name dpo_hh --baseline_name sft --seed 42
Running: 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\.venv-pa2\Scripts\python.exe' 'D:\projects\Assignments\CS5326\PA2\cs5326-pa2\scripts\build_pairs.py' --candidate_path outputs/generations/lora_persona.json --baseline_path outputs/generations/base_persona.jso

In [15]:
from zipfile import ZIP_DEFLATED, ZipFile


def create_selected_archive(archive_path, roots):
    archive_path = Path(archive_path)
    with ZipFile(archive_path, "w", compression=ZIP_DEFLATED, compresslevel=6) as archive:
        for root in roots:
            root = Path(root)
            if not root.exists():
                continue
            if root.is_file():
                archive.write(root, root.relative_to(REPO_ROOT).as_posix())
                continue
            for path in sorted(root.rglob("*")):
                if path.is_file():
                    archive.write(path, path.relative_to(REPO_ROOT).as_posix())
    print(f"Saved {archive_path} ({archive_path.stat().st_size / 2**20:.1f} MiB)")


create_selected_archive(
    REPO_ROOT / "generation_outputs.zip",
    [OUTPUTS / "generations", OUTPUTS / "eval", OUTPUTS / "sanity", REPO_ROOT / "eval_sets"],
)

Saved D:\projects\Assignments\CS5326\PA2\cs5326-pa2\generation_outputs.zip (0.7 MiB)
